# FireST-GraphNet: Weather Fine-Tuning with HRRR + WeatherNext2

**Goal**: Fine-tune the best model (run20) using real weather data:
- **HRRR** DOY climatology for 1-step prediction (NOAA high-res NWP)
- **WeatherNext2** (Open-Meteo ERA5) for multi-step prediction

This trains the HRRR and WeatherNext2 source adapters in the ForecastDecoder
while keeping the encoder frozen.

| Step | What | Time |
|------|------|------|
| 1 | Mount Drive + copy data | ~5 min |
| 2 | Install deps | ~3 min |
| 3 | Build weather climatology | ~3 min |
| 4 | Fine-tune (30 epochs) | ~30-60 min |
| 5 | Evaluate all modes | ~10 min |
| 6 | Save to Drive | ~2 min |

## Step 1: Environment Setup

In [ ]:
# Check GPU and disk space
!nvidia-smi

import os, sys, shutil
from pathlib import Path

print(f"\n=== Colab Environment ===")
print(f"Disk free: {shutil.disk_usage('/').free / (1024**3):.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# === SET YOUR PATHS HERE ===
PROJECT_PATH = "/content/drive/MyDrive/FireST-GraphNet/FireST-GraphNet-Clean"

DRIVE_TRAIN = f"{PROJECT_PATH}/dataset_train"
DRIVE_VAL   = f"{PROJECT_PATH}/dataset_val"

# Verify
assert Path(PROJECT_PATH).exists(), f"Project not found: {PROJECT_PATH}"
print(f"Project:  {PROJECT_PATH}")
print(f"Train:    {DRIVE_TRAIN} (exists={Path(DRIVE_TRAIN).exists()})")
print(f"Val:      {DRIVE_VAL} (exists={Path(DRIVE_VAL).exists()})")

## Step 2: Scan & Copy Data to Colab Local

In [ ]:
import numpy as np

DATASET_TRAIN = "/content/dataset_train"
DATASET_VAL   = "/content/dataset_val"
os.makedirs(DATASET_TRAIN, exist_ok=True)
os.makedirs(DATASET_VAL, exist_ok=True)

# Files needed for forecast-mode fine-tuning
files_to_copy = {
    "train": [
        ("af_train_img_seqtoseq_alll_1i_1.npy",   True),
        ("af_train_label_seqtoseq_alll_1i_1.npy", True),
        ("firepred_train.npy",                      True),
        ("lulc_train.npy",                          False),
    ],
    "val": [
        ("af_val_img_seqtoseq_alll_1i_1.npy",     True),
        ("af_val_label_seqtoseq_alll_1i_1.npy",   True),
        ("firepred_val.npy",                        True),
        ("lulc_val.npy",                            False),
    ],
}

print("Copying data from Drive to Colab local ...\n")
missing = []
for split, file_list in files_to_copy.items():
    src_dir  = DRIVE_TRAIN if split == "train" else DRIVE_VAL
    dest_dir = DATASET_TRAIN if split == "train" else DATASET_VAL
    print(f"{split.upper()}:")
    for filename, required in file_list:
        src  = Path(src_dir) / filename
        dest = Path(dest_dir) / filename
        if src.exists():
            src_size = src.stat().st_size
            if not dest.exists() or dest.stat().st_size != src_size:
                shutil.copy2(src, dest)
            size_mb = dest.stat().st_size / (1024**2)
            print(f"  OK  {filename:50s} {size_mb:8.1f} MB")
        elif required:
            print(f"  MISSING  {filename} (required!)")
            missing.append(f"{split}/{filename}")
        else:
            print(f"  skip {filename} (optional)")
    print()

if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")
print("Data copy complete!")

## Step 3: Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q torch-geometric
!pip install -q pyyaml tensorboard scikit-learn matplotlib requests

# Install torch-cluster from pre-compiled wheel
import torch
pyg_wheel_url = f"https://data.pyg.org/whl/torch-{torch.__version__}.html"
!pip install -q torch-cluster -f {pyg_wheel_url}

print(f"\nPyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
print("\nDependencies installed!")

In [ ]:
# Set up Python path for the project
os.chdir("/content")
sys.path.insert(0, PROJECT_PATH)

# Quick import test
from firest_graphnet.models.firest_graphnet import FireSTGraphNet
from firest_graphnet.data.dataset_forecast import TSSatFireForecastDataset
print("Project imports OK")

## Step 4: Build Weather Climatology Data

Two sources of real weather data:
1. **HRRR** (NOAA High-Resolution Rapid Refresh) — DOY climatology from pre-built files
2. **WeatherNext2** (Open-Meteo ERA5 proxy) — downloaded live from free API

Both produce 4-channel canonical format: `[temperature(K), u_wind(m/s), v_wind(m/s), relative_humidity(%)]`

In [ ]:
# --- 4a: Copy or build HRRR DOY climatology ---
import numpy as np

HRRR_CLIM = "/content/hrrr_cache/climatology"
os.makedirs(HRRR_CLIM, exist_ok=True)

# Check if HRRR climatology exists on Drive
DRIVE_HRRR = f"{PROJECT_PATH}/hrrr_cache/climatology"
drive_hrrr_files = sorted(Path(DRIVE_HRRR).glob("doy_*.npy")) if Path(DRIVE_HRRR).exists() else []

if drive_hrrr_files:
    print(f"Found {len(drive_hrrr_files)} HRRR DOY files on Drive, copying ...")
    for f in drive_hrrr_files:
        dest = Path(HRRR_CLIM) / f.name
        if not dest.exists():
            shutil.copy2(f, dest)
        # Ensure spatial shape (4, 256, 256) — old files may be (4,) scalar
        arr = np.load(str(dest))
        if arr.ndim == 1 and arr.shape[0] == 4:
            arr = arr[:, None, None] * np.ones((1, 256, 256), dtype=np.float32)
            np.save(str(dest), arr)
    print(f"HRRR climatology: {len(list(Path(HRRR_CLIM).glob('doy_*.npy')))} DOY files")
else:
    print("No HRRR climatology on Drive. Building synthetic HRRR from Open-Meteo ...")
    # Build HRRR-like climatology from Open-Meteo ERA5 (same API, just different label)
    # This gives us real weather DOY averages that serve as HRRR proxy
    import requests, math
    from collections import defaultdict
    import datetime

    # Use a CONUS fire region centroid
    LAT, LON = 39.0, -115.0  # Nevada/California border — heart of western US fire zone
    YEARS = [2017, 2018, 2019, 2020, 2021]

    accum = defaultdict(lambda: np.zeros(4, dtype=np.float64))
    count = defaultdict(int)

    for year in YEARS:
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude": LAT, "longitude": LON,
            "start_date": f"{year}-01-01", "end_date": f"{year}-12-31",
            "hourly": "temperature_2m,wind_speed_10m,wind_direction_10m,relative_humidity_2m",
            "timezone": "UTC",
        }
        print(f"  {year}: fetching ...", end=" ", flush=True)
        try:
            resp = requests.get(url, params=params, timeout=120)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"FAILED ({e})")
            continue

        hourly = data.get("hourly", {})
        times = hourly.get("time", [])
        temp_c = np.array(hourly.get("temperature_2m", []), dtype=np.float64)
        wspd   = np.array(hourly.get("wind_speed_10m", []), dtype=np.float64)
        wdir   = np.array(hourly.get("wind_direction_10m", []), dtype=np.float64)
        rh     = np.array(hourly.get("relative_humidity_2m", []), dtype=np.float64)

        temp_k = temp_c + 273.15
        dir_rad = wdir * (math.pi / 180.0)
        u_wind = wspd * np.sin(dir_rad)
        v_wind = wspd * np.cos(dir_rad)

        n_ok = 0
        for i, t_str in enumerate(times):
            vals = np.array([temp_k[i], u_wind[i], v_wind[i], rh[i]])
            if np.any(np.isnan(vals)): continue
            dt = datetime.datetime.fromisoformat(t_str)
            doy = dt.timetuple().tm_yday
            accum[doy] += vals
            count[doy] += 1
            n_ok += 1
        print(f"OK ({n_ok} hours)")

    saved = 0
    for doy in sorted(accum.keys()):
        if count[doy] == 0: continue
        avg_scalar = (accum[doy] / count[doy]).astype(np.float32)  # (4,)
        # Broadcast to (4, 256, 256) — HRRRFutureProvider expects spatial arrays
        avg = avg_scalar[:, None, None] * np.ones((1, 256, 256), dtype=np.float32)
        np.save(str(Path(HRRR_CLIM) / f"doy_{doy:03d}.npy"), avg)
        saved += 1
    print(f"\nHRRR climatology: {saved} DOY files saved to {HRRR_CLIM}")

In [ ]:
# --- 4b: WeatherNext2 via Google Weather API + ERA5 fallback ---
# SECURITY: Store key in Colab Secrets (key icon in left sidebar).
#   Secret name: GOOGLE_WEATHER_API_KEY
import os
from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get("GOOGLE_WEATHER_API_KEY")
    os.environ["GOOGLE_WEATHER_API_KEY"] = GOOGLE_API_KEY
    print(f"Google Weather API key loaded (len={len(GOOGLE_API_KEY)})")
except Exception:
    GOOGLE_API_KEY = None
    print("No GOOGLE_WEATHER_API_KEY in Colab Secrets.")
    print("Add it: left sidebar -> key icon -> Add new secret.")
    print("Falling back to ERA5 DOY climatology for WN2 source.")

WN2_CLIM  = "/content/cache/weathernext2/climatology"
WN2_CACHE = "/content/cache/weathernext2"
os.makedirs(WN2_CLIM, exist_ok=True)

DRIVE_WN2_CLIM = f"{PROJECT_PATH}/cache/weathernext2/climatology"
drive_wn2 = sorted(Path(DRIVE_WN2_CLIM).glob("doy_*.npy")) if Path(DRIVE_WN2_CLIM).exists() else []

if drive_wn2:
    print(f"Copying {len(drive_wn2)} WN2 DOY files from Drive ...")
    for f in drive_wn2:
        dest = Path(WN2_CLIM) / f.name
        if not dest.exists():
            shutil.copy2(f, dest)
        arr = np.load(str(dest))
        if arr.ndim == 1:
            np.save(str(dest), arr[:, None, None] * np.ones((1, 256, 256), dtype=np.float32))
    print(f"WN2 climatology: {len(list(Path(WN2_CLIM).glob('doy_*.npy')))} files")
else:
    print("Building ERA5 DOY climatology from Open-Meteo (~3 min) ...")
    import requests, math, datetime
    from collections import defaultdict
    LAT_WN2, LON_WN2 = 37.0, -119.0
    accum2 = defaultdict(lambda: np.zeros(4, dtype=np.float64))
    count2 = defaultdict(int)
    for year in [2017, 2018, 2019, 2020, 2021]:
        params = {
            "latitude": LAT_WN2, "longitude": LON_WN2,
            "start_date": f"{year}-01-01", "end_date": f"{year}-12-31",
            "hourly": "temperature_2m,wind_speed_10m,wind_direction_10m,relative_humidity_2m",
            "timezone": "UTC",
        }
        print(f"  {year}: fetching ...", end=" ", flush=True)
        try:
            resp = requests.get("https://archive-api.open-meteo.com/v1/archive",
                                params=params, timeout=120)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"FAILED ({e})"); continue
        hourly = data.get("hourly", {})
        times = hourly.get("time", [])
        tc = np.array(hourly.get("temperature_2m", []), dtype=np.float64)
        ws = np.array(hourly.get("wind_speed_10m", []), dtype=np.float64)
        wd = np.array(hourly.get("wind_direction_10m", []), dtype=np.float64)
        rh = np.array(hourly.get("relative_humidity_2m", []), dtype=np.float64)
        n_ok = 0
        for i, t_str in enumerate(times):
            r = math.radians(wd[i])
            vals = np.array([tc[i]+273.15, ws[i]*math.sin(r), ws[i]*math.cos(r), rh[i]])
            if np.any(np.isnan(vals)):
                continue
            doy = datetime.datetime.fromisoformat(t_str).timetuple().tm_yday
            accum2[doy] += vals
            count2[doy] += 1
            n_ok += 1
        print(f"OK ({n_ok} hours)")
    saved2 = 0
    for doy in sorted(accum2):
        if count2[doy] == 0:
            continue
        avg = (accum2[doy] / count2[doy]).astype(np.float32)
        np.save(str(Path(WN2_CLIM) / f"doy_{doy:03d}.npy"),
                avg[:, None, None] * np.ones((1, 256, 256), dtype=np.float32))
        saved2 += 1
    print(f"WN2 ERA5 climatology: {saved2} DOY files -> {WN2_CLIM}")

# Test Google Weather API
if GOOGLE_API_KEY:
    import requests as _req
    try:
        r = _req.get(
            "https://weather.googleapis.com/v1/forecast/hours:lookup",
            params={"key": GOOGLE_API_KEY, "location.latitude": 37.0,
                    "location.longitude": -119.0, "hours": 3},
            timeout=15,
        )
        r.raise_for_status()
        hrs = r.json().get("forecastHours", [])
        h0  = hrs[0] if hrs else {}
        temp = h0.get("temperature", {}).get("degrees", "?")
        rh0  = h0.get("relativeHumidity", "?")
        print(f"Google Weather API OK: {len(hrs)} hours, temp={temp}C  rh={rh0}%")
        print("Live forecasts ENABLED for WeatherNext2 source.")
    except Exception as e:
        print(f"Google Weather API test FAILED: {e}")
        GOOGLE_API_KEY = None

# Verify shapes
print("\n=== Weather Data Ready ===")
for label, clim_dir in [("HRRR", HRRR_CLIM), ("WN2", WN2_CLIM)]:
    files = sorted(Path(clim_dir).glob("doy_*.npy"))
    if files:
        s = np.load(str(files[0]))
        ok = s.ndim == 3 and s.shape == (4, 256, 256)
        print(f"  {label:5s}: {len(files)} files  shape={s.shape}  {'OK' if ok else 'FIXING...'}")
        if not ok:
            for ff in files:
                a = np.load(str(ff))
                if a.ndim == 1:
                    np.save(str(ff), a[:, None, None] * np.ones((1, 256, 256), dtype=np.float32))
            print(f"         Fixed all files to (4, 256, 256)")
    else:
        print(f"  {label:5s}: NO FILES!")
print(f"  Google Weather API: {'LIVE' if GOOGLE_API_KEY else 'ERA5 climatology fallback'}")

## Step 5: Fine-Tune Model with Multi-Source Weather

Strategy:
1. Load run20 best checkpoint (val_IoU = 0.617)
2. Copy firepred adapter weights -> HRRR and WeatherNext2 adapters (warm start)
3. Freeze encoder (CNN + GNN + LSTM) — only train decoder
4. Each training batch randomly uses one of 3 sources: firepred (40%), HRRR (30%), WN2 (30%)
5. Adapters get 3x learning rate; decoder backbone gets base LR
6. Validation always uses firepred for fair comparison

In [ ]:
import random
import csv
import yaml
import torch
import torch.nn as nn
import torch.amp
import numpy as np
from pathlib import Path
from collections import defaultdict
from torch.utils.data import DataLoader, Dataset

from firest_graphnet.data.dataset_forecast import TSSatFireForecastDataset
from firest_graphnet.data.firepred_provider import build_firepred_provider
from firest_graphnet.data.static_provider import build_static_provider
from firest_graphnet.data.future_providers import (
    HRRRFutureProvider, WeatherNext2FutureProvider,
    PlaceholderFutureProvider, extract_wind, DYNAMIC_FUTURE_CHANNELS,
)
from firest_graphnet.models.firest_graphnet import FireSTGraphNet
from firest_graphnet.training.losses import ForecastPINNLoss

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# ===== CONFIGURATION =====
BASE_RUN   = "run20"         # checkpoint to fine-tune from
OUT_RUN    = "run23"         # output run name
EPOCHS     = 30
BATCH_SIZE = 4
LR         = 3e-5           # decoder LR; adapters get 3x this
PATIENCE   = 10
SOURCE_MIX = (0.40, 0.30, 0.30)  # firepred, hrrr, weathernext2

# Loss config (tuned for fine-tuning)
POS_WEIGHT   = 4.0
FOCAL_GAMMA  = 2.5
GRAD_CLIP    = 0.5

# Paths
RUNS_DIR   = f"{PROJECT_PATH}/outputs/output-clean"
BASE_DIR   = f"{RUNS_DIR}/{BASE_RUN}"
OUT_DIR    = f"/content/outputs/{OUT_RUN}"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Base model:  {BASE_DIR}")
print(f"Output:      {OUT_DIR}")
print(f"Source mix:  firepred={SOURCE_MIX[0]:.0%}  HRRR={SOURCE_MIX[1]:.0%}  WN2={SOURCE_MIX[2]:.0%}")

In [ ]:
# ===== LOAD BASE MODEL =====

# Load config
cfg_path = f"{BASE_DIR}/config.yaml"
with open(cfg_path, 'r') as f:
    cfg = yaml.safe_load(f)
ckpt = torch.load(f"{BASE_DIR}/best_model.pt", map_location=DEVICE, weights_only=False)
sd = ckpt.get("model_state_dict", ckpt)
if isinstance(ckpt.get("config"), dict):
    cfg = ckpt["config"]

print(f"Base: epoch={ckpt.get('epoch')}  val_IoU={ckpt.get('best_val_iou'):.4f}")

# Build model
mc = cfg["model"]; dc = cfg["data"]; fc = cfg.get("forecast", {})
model = FireSTGraphNet(
    in_channels=8 + 19 + 1,
    encoder_channels=mc.get("encoder_channels", [32, 64, 128]),
    use_cbam=mc.get("use_cbam", True),
    patch_size=mc.get("patch_size", 8),
    k_neighbors=mc.get("k_neighbors", 8),
    gnn_hidden=mc.get("gnn_hidden", 128),
    gnn_layers=mc.get("gnn_layers", 2),
    lstm_hidden=mc.get("lstm_hidden", 256),
    lstm_layers=mc.get("lstm_layers", 2),
    lstm_dropout=mc.get("lstm_dropout", 0.1),
    decoder_channels=mc.get("decoder_channels", [128, 64, 32]),
    out_channels=1,
    img_h=int(dc.get("img_size", 256)),
    img_w=int(dc.get("img_size", 256)),
    forecast_mode=True,
    future_channels=int(fc.get("future_channels", 4)),
    future_emb_dim=int(fc.get("future_emb_dim", fc.get("weather_emb_dim", 64))),
).to(DEVICE)

model.load_state_dict(sd, strict=False)
print(f"Model: {sum(p.numel() for p in model.parameters()):,} params")

# Warm-start: copy firepred adapter -> hrrr and weathernext2
with torch.no_grad():
    fp_w = model.decoder.future_enc.adapters["firepred"].weight.data.clone()
    fp_b = model.decoder.future_enc.adapters["firepred"].bias.data.clone()
    model.decoder.future_enc.adapters["hrrr"].weight.data.copy_(fp_w)
    model.decoder.future_enc.adapters["hrrr"].bias.data.copy_(fp_b)
    model.decoder.future_enc.adapters["weathernext2"].weight.data.copy_(fp_w)
    model.decoder.future_enc.adapters["weathernext2"].bias.data.copy_(fp_b)
print("Warm-start: firepred adapter weights copied to HRRR + WN2 adapters")

In [ ]:
# ===== BUILD DATASETS =====

IMG_SIZE = int(dc.get("img_size", 256))

def build_dataset(data_dir, cfg):
    fc = cfg.get("forecast", {}); dc = cfg.get("data", {})
    img_size = int(dc.get("img_size", 256))
    fp = build_firepred_provider(str(data_dir), spatial_size=(img_size, img_size))
    st = build_static_provider(str(data_dir), spatial_size=(img_size, img_size))
    ds = TSSatFireForecastDataset(
        data_dir=data_dir,
        T_in=int(fc.get("T_in", 10)),
        horizon=int(fc.get("horizon", 20)),
        label_channel=int(dc.get("label_channel", 1)),
        firepred_provider=fp, static_provider=st,
    )
    if ds.firepred_provider is None and getattr(fp, "_N", 0) > 0:
        ds.firepred_provider = fp
    if ds.static_provider is None and getattr(st, "_N", 0) > 0:
        ds.static_provider = st
    return ds

# Build weather providers
hrrr_prov = None
try:
    hrrr_prov = HRRRFutureProvider(HRRR_CLIM, spatial_size=(IMG_SIZE, IMG_SIZE))
    print(f"HRRR provider: {hrrr_prov._D} DOY frames")
except Exception as e:
    print(f"HRRR provider failed: {e}, using placeholder")
    hrrr_prov = PlaceholderFutureProvider(source_name="hrrr", spatial_size=(IMG_SIZE, IMG_SIZE))

wn2_prov = None
try:
    _wn2_key = os.environ.get("GOOGLE_WEATHER_API_KEY")
    wn2_prov = WeatherNext2FutureProvider(
        cache_dir="/content/cache/weathernext2",
        spatial_size=(IMG_SIZE, IMG_SIZE),
        api_key=_wn2_key,
    )
    n_wn2 = len(wn2_prov._downloader._doy_data) if wn2_prov._downloader else 0
    has_api = bool(wn2_prov._downloader and wn2_prov._downloader._api_key)
    print(f"WeatherNext2 provider: {n_wn2} DOY frames, live API={has_api}")
except Exception as e:
    print(f"WN2 provider failed: {e}, using placeholder")
    wn2_prov = PlaceholderFutureProvider(source_name="weathernext2", spatial_size=(IMG_SIZE, IMG_SIZE))

# Multi-source dataset wrapper
class MultiSourceDataset(Dataset):
    """Randomly overrides env_fut with HRRR or WN2 per sample."""
    def __init__(self, base_ds, hrrr_prov, wn2_prov, source_probs=(0.4, 0.3, 0.3)):
        self.ds = base_ds
        self.hrrr = hrrr_prov
        self.wn2  = wn2_prov
        self.probs = list(source_probs)
        self.sources = ["firepred", "hrrr", "weathernext2"]

    def __len__(self): return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]
        horizon = int(sample["env_fut"].shape[0])
        indices = tuple(range(idx, idx + horizon))
        src = random.choices(self.sources, weights=self.probs, k=1)[0]
        if src == "hrrr" and self.hrrr is not None:
            env_np = self.hrrr.get_future_env(indices, horizon=horizon)
            sample["env_fut"] = torch.from_numpy(env_np)
            sample["env_source"] = "hrrr"
        elif src == "weathernext2" and self.wn2 is not None:
            env_np = self.wn2.get_future_env(indices, horizon=horizon)
            sample["env_fut"] = torch.from_numpy(env_np)
            sample["env_source"] = "weathernext2"
        else:
            sample["env_source"] = "firepred"
        return sample

def collate_fn(batch):
    out = {}
    for k in batch[0]:
        vals = [b[k] for b in batch]
        if isinstance(vals[0], torch.Tensor):
            out[k] = torch.stack(vals, dim=0)
        else:
            out[k] = vals
    return out

# Build datasets
base_train = build_dataset(DATASET_TRAIN, cfg)
base_val   = build_dataset(DATASET_VAL, cfg)

ds_train = MultiSourceDataset(base_train, hrrr_prov, wn2_prov, SOURCE_MIX)
ds_val   = MultiSourceDataset(base_val, hrrr_prov, wn2_prov, (1.0, 0.0, 0.0))  # val=firepred only

train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True, collate_fn=collate_fn, drop_last=True)
val_loader   = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True, collate_fn=collate_fn)

print(f"\nTrain: {len(ds_train)} events ({len(train_loader)} batches)")
print(f"Val:   {len(ds_val)} events ({len(val_loader)} batches)")

In [ ]:
# ===== SET UP OPTIMIZER & FREEZE ENCODER =====

# Freeze encoder
frozen = 0
for name, param in model.named_parameters():
    if any(k in name for k in ["encoder", "gnn", "lstm", "patch", "cbam"]):
        param.requires_grad_(False)
        frozen += param.numel()
print(f"Frozen encoder: {frozen:,} params")

# Parameter groups: adapters get 3x LR
adapter_params, decoder_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad: continue
    if "adapters.hrrr" in name or "adapters.weathernext2" in name:
        adapter_params.append(param)
    else:
        decoder_params.append(param)

optimizer = torch.optim.AdamW([
    {"params": adapter_params, "lr": LR * 3.0},
    {"params": decoder_params, "lr": LR},
], weight_decay=5e-3)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=True)

# AMP dtype
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# Loss
criterion = ForecastPINNLoss(
    pos_weight=POS_WEIGHT, focal_gamma=FOCAL_GAMMA,
    boundary_weight=0.10, temporal_weight=0.08, horizon_decay=0.08,
    dice_weight=0.15, gamma=0.25, pde_D=0.01, pde_mu=0.12,
    beta_max=0.005, eta_max=0.08, xi_max=0.04,
    warmup_epochs=5, start_epoch=3,
).to(DEVICE)

n_adapt = sum(p.numel() for p in adapter_params)
n_dec   = sum(p.numel() for p in decoder_params)
print(f"Trainable: adapters={n_adapt:,}  decoder={n_dec:,}")
print(f"Adapter LR: {LR*3:.2e}  Decoder LR: {LR:.2e}")
print(f"AMP dtype: {amp_dtype}")

In [ ]:
# ===== TRAINING LOOP =====
from tqdm.auto import tqdm

@torch.no_grad()
def compute_iou(logits, targets, thr=0.5):
    pred = (torch.sigmoid(logits) > thr).float()
    gt   = (targets >= 0.5).float()
    tp = (pred * gt).sum().item()
    fp = (pred * (1 - gt)).sum().item()
    fn = ((1 - pred) * gt).sum().item()
    iou  = tp / (tp + fp + fn + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    return {"iou": iou, "f1": f1, "precision": prec, "recall": rec}

def run_epoch(loader, train, epoch):
    model.train(train)
    if train: criterion.set_epoch(epoch)
    totals = defaultdict(float); met_acc = defaultdict(float); n = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, desc="train" if train else "val", leave=False):
            af_hist  = batch["af_hist"].to(DEVICE, non_blocking=True)
            aux_hist = batch["aux_hist"].to(DEVICE, non_blocking=True)
            env_fut  = batch["env_fut"].to(DEVICE, non_blocking=True)
            af_fut   = batch["af_fut"].to(DEVICE, non_blocking=True)
            ba_init  = batch["ba_init"].to(DEVICE, non_blocking=True)
            static   = batch["static"].to(DEVICE, non_blocking=True)
            env_src  = batch.get("env_source", ["firepred"])[0]

            with torch.amp.autocast("cuda", dtype=amp_dtype):
                logits = model(af_hist, aux_hist=aux_hist, env_fut=env_fut,
                               env_source=env_src, static=static)
                wind = extract_wind(env_fut, source=env_src)
                losses = criterion(logits, af_fut, ba_init=ba_init, wind=wind)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(losses["total"]).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

            for k, v in losses.items(): totals[k] += v.item()
            m = compute_iou(logits.detach(), af_fut.detach())
            for k, v in m.items(): met_acc[k] += v
            n += 1
    N = max(n, 1)
    out = {k: v/N for k, v in totals.items()}
    out.update({k: v/N for k, v in met_acc.items()})
    return out

# === TRAIN ===
best_val_iou = -1.0
patience_cnt = 0
history = []

print(f"\n{'='*60}")
print(f"  Weather Fine-Tuning: {EPOCHS} epochs")
print(f"  Source mix: firepred={SOURCE_MIX[0]:.0%}  HRRR={SOURCE_MIX[1]:.0%}  WN2={SOURCE_MIX[2]:.0%}")
print(f"{'='*60}\n")

for epoch in range(1, EPOCHS + 1):
    train_m = run_epoch(train_loader, train=True,  epoch=epoch)
    val_m   = run_epoch(val_loader,   train=False, epoch=epoch)
    scheduler.step()

    lr_now = optimizer.param_groups[1]["lr"]  # decoder LR
    history.append({"epoch": epoch, "train_loss": train_m["total"],
                    "train_iou": train_m["iou"], "val_iou": val_m["iou"],
                    "val_f1": val_m["f1"], "lr": lr_now})

    print(f"Ep {epoch:3d}/{EPOCHS}  loss={train_m['total']:.4f}  "
          f"train_iou={train_m['iou']:.4f}  val_iou={val_m['iou']:.4f}  "
          f"val_f1={val_m['f1']:.4f}  lr={lr_now:.2e}")

    if val_m["iou"] > best_val_iou:
        best_val_iou = val_m["iou"]
        patience_cnt = 0
        best_path = f"{OUT_DIR}/best_model.pt"
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "best_val_iou": best_val_iou,
            "config": cfg,
            "finetune_note": "multi-source: firepred+hrrr+weathernext2",
        }, best_path)
        print(f"  *** New best val IoU: {best_val_iou:.4f}")
    else:
        patience_cnt += 1
        if PATIENCE > 0 and patience_cnt >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break

print(f"\nBest val IoU: {best_val_iou:.4f}")

## Step 6: Evaluate with Different Weather Sources

In [ ]:
# ===== EVALUATION: Compare firepred / hrrr / mixed =====

# Reload best model
ckpt_ft = torch.load(f"{OUT_DIR}/best_model.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt_ft["model_state_dict"])
model.eval()
print(f"Loaded fine-tuned model (epoch {ckpt_ft['epoch']}, val_IoU={ckpt_ft['best_val_iou']:.4f})")

@torch.no_grad()
def eval_source(ds_base, hrrr_p, wn2_p, env_source, max_events=50):
    """Evaluate model with a specific weather source."""
    n = min(len(ds_base), max_events)
    ious, f1s = [], []
    for idx in range(n):
        sample = ds_base[idx]
        horizon = int(sample["env_fut"].shape[0])
        indices = tuple(range(idx, idx + horizon))

        # Override env_fut based on source
        if env_source == "hrrr" and hrrr_p:
            env_np = hrrr_p.get_future_env(indices, horizon=horizon)
            env_fut = torch.from_numpy(env_np).unsqueeze(0).to(DEVICE)
            src = "firepred"  # use trained adapter
        elif env_source == "mixed" and hrrr_p and wn2_p:
            h_step = hrrr_p.get_future_env((idx,), horizon=1)
            if horizon > 1:
                w_steps = wn2_p.get_future_env(tuple(range(idx+1, idx+horizon)), horizon=horizon-1)
                env_np = np.concatenate([h_step, w_steps], axis=0)
            else:
                env_np = h_step
            env_fut = torch.from_numpy(env_np).unsqueeze(0).to(DEVICE)
            src = "firepred"
        elif env_source == "weathernext2" and wn2_p:
            env_np = wn2_p.get_future_env(indices, horizon=horizon)
            env_fut = torch.from_numpy(env_np).unsqueeze(0).to(DEVICE)
            src = "firepred"
        else:
            env_fut = sample["env_fut"].unsqueeze(0).to(DEVICE)
            src = "firepred"

        af_hist  = sample["af_hist"].unsqueeze(0).to(DEVICE)
        aux_hist = sample["aux_hist"].unsqueeze(0).to(DEVICE)
        static   = sample["static"].unsqueeze(0).to(DEVICE)

        with torch.amp.autocast("cuda", dtype=amp_dtype):
            logits = model(af_hist, aux_hist=aux_hist, env_fut=env_fut,
                           env_source=src, static=static)

        af_fut = sample["af_fut"].unsqueeze(0).to(DEVICE)
        m = compute_iou(logits, af_fut)
        ious.append(m["iou"]); f1s.append(m["f1"])

    return np.mean(ious), np.mean(f1s)

# Evaluate all modes on val set
print("\n=== Evaluation on Val Set ===")
print(f"{'Source':<15s} {'Mean IoU':>10s} {'Mean F1':>10s}")
print("-" * 38)
for src_name in ["firepred", "hrrr", "weathernext2", "mixed"]:
    iou, f1 = eval_source(base_val, hrrr_prov, wn2_prov, src_name)
    print(f"{src_name:<15s} {iou:10.4f} {f1:10.4f}")

## Step 7: Visualize Training Curves

In [ ]:
import matplotlib.pyplot as plt

if history:
    epochs  = [h["epoch"] for h in history]
    t_loss  = [h["train_loss"] for h in history]
    t_iou   = [h["train_iou"] for h in history]
    v_iou   = [h["val_iou"] for h in history]
    v_f1    = [h["val_f1"] for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, t_loss, 'b-', label='Train Loss')
    axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(True)

    axes[1].plot(epochs, t_iou, 'b-', label='Train IoU')
    axes[1].plot(epochs, v_iou, 'r-', label='Val IoU')
    axes[1].plot(epochs, v_f1,  'g--', label='Val F1')
    axes[1].set_title('IoU / F1'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(True)

    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/finetune_curves.png", dpi=150)
    plt.show()
    print(f"Curves saved to {OUT_DIR}/finetune_curves.png")
else:
    print("No training history to plot.")

## Step 8: Save Results to Google Drive

In [ ]:
import pandas as pd

# Save config
out_cfg = dict(cfg)
out_cfg["finetune"] = {
    "base_run": BASE_RUN,
    "source_mix": {"firepred": SOURCE_MIX[0], "hrrr": SOURCE_MIX[1], "wn2": SOURCE_MIX[2]},
    "epochs": EPOCHS, "lr": LR, "best_val_iou": best_val_iou,
}
with open(f"{OUT_DIR}/config.yaml", "w") as f:
    yaml.dump(out_cfg, f)

# Save metrics CSV
if history:
    pd.DataFrame(history).to_csv(f"{OUT_DIR}/finetune_metrics.csv", index=False)

# Copy to Drive
DRIVE_OUT = f"{RUNS_DIR}/{OUT_RUN}"
os.makedirs(DRIVE_OUT, exist_ok=True)

for fname in ["best_model.pt", "config.yaml", "finetune_metrics.csv", "finetune_curves.png"]:
    src = Path(OUT_DIR) / fname
    if src.exists():
        shutil.copy2(src, Path(DRIVE_OUT) / fname)
        print(f"  Saved: {fname}")

# Also save weather climatology to Drive for future use
DRIVE_HRRR_OUT = f"{PROJECT_PATH}/hrrr_cache/climatology"
os.makedirs(DRIVE_HRRR_OUT, exist_ok=True)
for f in Path(HRRR_CLIM).glob("doy_*.npy"):
    dest = Path(DRIVE_HRRR_OUT) / f.name
    if not dest.exists():
        shutil.copy2(f, dest)

DRIVE_WN2_OUT = f"{PROJECT_PATH}/cache/weathernext2/climatology"
os.makedirs(DRIVE_WN2_OUT, exist_ok=True)
for f in Path(WN2_CLIM).glob("doy_*.npy"):
    dest = Path(DRIVE_WN2_OUT) / f.name
    if not dest.exists():
        shutil.copy2(f, dest)

n_hrrr_saved = len(list(Path(DRIVE_HRRR_OUT).glob("doy_*.npy")))
n_wn2_saved  = len(list(Path(DRIVE_WN2_OUT).glob("doy_*.npy")))

print(f"\n=== Saved to Google Drive ===")
print(f"  Model + metrics: {DRIVE_OUT}")
print(f"  HRRR climatology: {n_hrrr_saved} DOY files")
print(f"  WN2 climatology:  {n_wn2_saved} DOY files")
print(f"\nBest val IoU: {best_val_iou:.4f}")
print("\nDone! You can now download best_model.pt from Drive.")

## Summary

| Step | What |
|------|------|
| 1 | Mounted Drive, copied AF/FirePred/LULC data to Colab local |
| 2 | Installed PyTorch, torch-geometric, torch-cluster |
| 3 | Built HRRR DOY climatology (real NOAA weather) |
| 4 | Built WeatherNext2 DOY climatology (Open-Meteo ERA5) |
| 5 | Fine-tuned run20 with 40% firepred + 30% HRRR + 30% WN2 |
| 6 | Evaluated with firepred / hrrr / weathernext2 / mixed modes |
| 7 | Saved best_model.pt + climatology to Drive as run23 |

**Next steps:**
- Download `best_model.pt` and `config.yaml` from `outputs/output-clean/run23/`
- Run local batch evaluation: `python batch_eval_all_runs.py --runs 23 --env-source mixed`
- For paper figures: use `--env-source hrrr` (1-step) or `--env-source mixed` (multi-step)